# Temel değerlendirme: aynı puan, daha az toplama işlemi

Yeni tarafta **EvalPackedBase=true**, kontrolde **false**. Her iki tarafta
**TTTwoContexts=true**, SEE kapalı ve NextIterationPct=150. Malzeme, kare
puanları, oyun evresi ve diğer değerlendirme terimleri değişmez. Yeni aday
main'e alınmaz; varsayılan kapalı kalır.

Önce tüm doğruluk testleri ve TT'nin açık olduğu main referansına karşı
sabit depth-4 eşdeğerliği kontrol edilir. Ardından düz/paketli, paketli/düz,
düz/paketli sırasıyla üçer hız ölçümü alınır. Bütün hamle, puan, derinlik ve
düğüm sonuçları aynı olmalı; paketli medyan süre en az %2 daha kısa olmalı.
Geçerse **100 ms/hamle, tek toplu 100 maç** başlar. Geçmezse maçlar atlanır,
ölçümler ve gerekçeler GitHub'a gönderilir. Bu ön kontrol Elo artışı kanıtı değildir.

`codex/packed-eval` dalından içe alın. Internet ve Contents yazma yetkili
`GITHUB_TOKEN` secret'ı açık olsun. Hazırlık ayrı Python sürecinde çalışır;
eski `tools` importları kullanılmaz. Bütün gerçek test/arama/maçlar Kaggle'dadır.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/packed-eval"
WORKSPACE = "/kaggle/working/acemi-packed-eval"
SESSION_ID = "packed-eval-100ms-100games-v1"
GAMES = 100
CHUNK_GAMES = 100
MOVETIME_MS = 100
HASH_MB = 16
CONCURRENCY = 2
MAX_PLIES = 400
SEED = 1
RETRY_INCOMPLETE = False
REFERENCE_COMMIT = "b22b8597dcdf40fa632d9585cd4901c87d97b306"
NEW_OPTIONS = {"EvalPackedBase": True, "TTTwoContexts": True, "QuiescenceSEE": False, "NextIterationPct": 150}
OLD_OPTIONS = {"EvalPackedBase": False, "TTTwoContexts": True, "QuiescenceSEE": False, "NextIterationPct": 150}


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This notebook runs matches only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
if not (workspace / "tools" / "ab_compare.py").exists():
    raise RuntimeError("Import the latest notebook from codex/packed-eval and rerun setup.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)

def pinned_checkout(path, commit):
    path = Path(path)
    if not path.exists():
        checked(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL, str(path)])
    checked(["git", "-C", str(path), "fetch", "--depth", "1", "origin", commit])
    checked(["git", "-C", str(path), "checkout", "--detach", commit])
    if checked(["git", "-C", str(path), "rev-parse", "HEAD"]) != commit:
        raise RuntimeError("Pinned source mismatch")
    return path

# The first run pins the full branch commit. Resume uses the saved source SHA,
# even after data-only commits have been published to the same branch.
session = workspace / "data" / "ab" / SESSION_ID
if (session / "protocol.json").exists():
    saved = json.loads((session / "protocol.json").read_text())
    CODE_COMMIT = saved["sources"]["new"]["commit"]
    if saved["sources"]["old"]["commit"] != CODE_COMMIT:
        raise RuntimeError("Packed-eval candidates must use the same source commit.")
else:
    CODE_COMMIT = checked(["git", "-C", str(workspace), "rev-parse", "HEAD"])
NEW_COMMIT = OLD_COMMIT = CODE_COMMIT
new_source = pinned_checkout("/kaggle/working/acemi-eval-candidate", CODE_COMMIT)
old_source = pinned_checkout("/kaggle/working/acemi-eval-control", CODE_COMMIT)
reference_source = pinned_checkout("/kaggle/working/acemi-eval-reference", REFERENCE_COMMIT)
print("PINNED SOURCE:", CODE_COMMIT)
print("NEW:", NEW_OPTIONS, "REFERENCE:", OLD_OPTIONS)
print("100 games in ONE run; 50 paired openings; 100 ms/move; results:", session)


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    paths += [p for p in ("data/runs", "data/index.csv", "data/positions") if (workspace / p).exists()]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Prepare in a fresh interpreter: old Kaggle kernel imports cannot select another tools package.
prepare_config = {
    "workspace": str(workspace), "session": str(session),
    "new_source": str(new_source), "old_source": str(old_source),
    "CODE_COMMIT": CODE_COMMIT, "GAMES": GAMES, "CHUNK_GAMES": CHUNK_GAMES,
    "MOVETIME_MS": MOVETIME_MS, "CONCURRENCY": CONCURRENCY, "HASH_MB": HASH_MB,
    "MAX_PLIES": MAX_PLIES, "SEED": SEED,
    "NEW_OPTIONS": NEW_OPTIONS, "OLD_OPTIONS": OLD_OPTIONS,
}
prepare_code = """
import json, os, sys
from pathlib import Path
config = json.loads(sys.argv[1])
workspace = Path(config.pop("workspace")).resolve()
sys.path.insert(0, str(workspace))
session = Path(config.pop("session"))
new_source = Path(config.pop("new_source"))
old_source = Path(config.pop("old_source"))
globals().update(config)
import inspect
from tools import ab_compare as ab
if Path(ab.__file__).resolve() != workspace / "tools" / "ab_compare.py":
    raise RuntimeError("Preparation loaded tools outside the selected workspace.")

if "new_commit" not in inspect.signature(ab.prepare).parameters:
    raise RuntimeError("Selected checkout lacks the required prepare API; rerun setup.")

if (session / "protocol.json").exists():
    plan = ab.load_plan(session)
    expected = {"games": GAMES, "movetime_ms": MOVETIME_MS, "hash_mb": HASH_MB,
                "max_plies": MAX_PLIES, "selection_seed": SEED, "chunk_games": CHUNK_GAMES,
                "concurrency": min(CONCURRENCY, os.cpu_count() or 1),
                "feature_options": {"new": NEW_OPTIONS, "old": OLD_OPTIONS}}
    if any(plan[key] != value for key, value in expected.items()):
        raise RuntimeError("Settings differ from saved experiment; use a new SESSION_ID.")
else:
    plan = ab.prepare(session, new_source, old_source, games=GAMES, chunk_games=CHUNK_GAMES,
                      movetime_ms=MOVETIME_MS, concurrency=CONCURRENCY, hash_mb=HASH_MB,
                      max_plies=MAX_PLIES, seed=SEED,
                      new_commit=CODE_COMMIT, old_commit=CODE_COMMIT,
                      new_options=NEW_OPTIONS, old_options=OLD_OPTIONS)
"""
checked([sys.executable, "-u", "-c", prepare_code, json.dumps(prepare_config)], cwd=workspace)
plan = json.loads((session / "protocol.json").read_text(encoding="utf-8"))
print((session / "SUMMARY.md").read_text(encoding="utf-8"))
publish_results(f"data: prepare packed-eval 100-game A/B ({SESSION_ID})")


In [ ]:
# Full correctness and repeated normal search timings; no kernel imports of tools.
command = [sys.executable, "-u", "-m", "tools.packed_eval_experiment", "--session", str(session),
           "--reference", str(reference_source)]
result = subprocess.run(command, cwd=workspace)
publish_results(f"data: packed-eval preflight records ({SESSION_ID})")
if result.returncode:
    raise RuntimeError("Packed-eval preflight failed; no matches started. Inspect published validation logs.")
print((session / "eligibility.json").read_text())


In [ ]:
# Verify the saved decision and every artifact again without rerunning completed searches.
checked([sys.executable, "-u", "-m", "tools.packed_eval_experiment", "--session", str(session),
         "--reference", str(reference_source)], cwd=workspace)
# Match eligibility is validated against immutable preflight inputs.
import hashlib
status = json.loads((session / "validation" / "status.json").read_text())
gate = json.loads((session / "eligibility.json").read_text())
if (not status.get("passed") or status["identity"].get("experiment") != "packed-eval"
        or status["identity"]["protocol_sha256"] != plan["protocol_sha256"]
        or gate["protocol_sha256"] != plan["protocol_sha256"]):
    raise RuntimeError("Preflight does not belong to this protocol.")
for path, digest in gate["artifacts"].items():
    if hashlib.sha256((workspace / path).read_bytes()).hexdigest() != digest:
        raise RuntimeError("Preflight artifact changed; do not start matches.")
if not gate["eligible_to_run_match"]:
    print("Matches SKIPPED: speed gate did not pass.")
    print(gate["reasons"])
elif len(plan["chunks"]) != 1 or plan["chunks"][0]["games"] != 100 or plan["movetime_ms"] != 100:
    raise RuntimeError("Use one bulk 100-game run at 100 ms.")
else:
    command = [sys.executable, "-u", "-m", "tools.ab_compare", "run", "--session", str(session), "--chunk", "1"]
    if RETRY_INCOMPLETE:
        command.append("--retry-incomplete")
    print("Starting ONE 100-game packed-eval on/off A/B: 100 ms/move for both sides.", flush=True)
    result = subprocess.run(command, cwd=workspace)
    publish_results(f"data: packed-eval 100-game results ({SESSION_ID})")
    if result.returncode:
        raise RuntimeError("Packed-eval A/B stopped; inspect saved logs before retrying.")
    print((session / "SUMMARY.md").read_text())


In [ ]:
# Safe to rerun after a push failure: it does not start any games.
publish_results(f"data: finalize packed-eval 100-game A/B records ({SESSION_ID})")
if (session / "results.json").exists():
    print((session / "SUMMARY.md").read_text())
    result = json.loads((session / "results.json").read_text())
    if result["recorded_attempts"] == GAMES:
        print("packed-eval 100-game A/B finished. W/D/L is from the EvalPackedBase=true side perspective.")
if (session / "eligibility.json").exists():
    gate = json.loads((session / "eligibility.json").read_text())
    if not gate["eligible_to_run_match"]:
        print("Preflight completed; 100 new matches were NOT played because the speed gate did not pass.")
